In [1]:
import pandas as pd

# Load RAW files
vendors = pd.read_csv("../data/raw/vendors.csv")
products = pd.read_csv("../data/raw/products.csv")
orders = pd.read_csv("../data/raw/purchase_orders.csv")
deliveries = pd.read_csv("../data/raw/deliveries.csv")

In [2]:
def missing_report(df, name):
    miss = df.isna().sum()          # count of missing values per column
    miss = miss[miss > 0]           # keep only columns that have missing values
    print(f"--- {name} ---")
    if miss.empty:
        print("No missing values")
    else:
        print(pd.DataFrame({"missing": miss,
                            "missing_%": (miss / len(df) * 100).round(2)}))

for name, df in [("vendors", vendors), ("products", products),
                 ("orders", orders), ("deliveries", deliveries)]:
    missing_report(df, name)

--- vendors ---
      missing  missing_%
city        2        8.0
--- products ---
No missing values
--- orders ---
            missing  missing_%
unit_price       40       0.66
--- deliveries ---
                      missing  missing_%
actual_delivery_date       20       0.34


In [3]:
print("Duplicate rows (orders):", orders.duplicated().sum())
orders = orders.drop_duplicates()
print("Shape after:", orders.shape)
print("po_id unique?", orders["po_id"].is_unique)

# Check the other tables too
for name, df in [("vendors", vendors), ("products", products), ("deliveries", deliveries)]:
    print(name, "duplicates:", df.duplicated().sum())

Duplicate rows (orders): 60
Shape after: (6000, 9)
po_id unique? True
vendors duplicates: 0
products duplicates: 0
deliveries duplicates: 0


In [4]:
mask = orders["unit_price"].isna()                      # True for the 40 missing rows

# Safety check: formula tabhi sahi hai jab quantity valid ho
print((orders.loc[mask, "quantity_ordered"] > 0).all())  # expect True

orders.loc[mask, "unit_price"] = (
    orders.loc[mask, "total_amount"] / orders.loc[mask, "quantity_ordered"]
).round(2)

print("Missing unit_price now:", orders["unit_price"].isna().sum())
orders.loc[mask].head(3)[["po_id", "quantity_ordered", "unit_price", "total_amount"]]

True
Missing unit_price now: 0


,po_id,quantity_ordered,unit_price,total_amount
308,PO004517,69,144.53,9972.57
401,PO005835,148,508.47,75253.56
431,PO002379,124,1072.04,132932.96


In [5]:
vendors["city"] = vendors["city"].fillna("Unknown")
print(vendors["city"].isna().sum())     # expect 0

0


In [6]:
deliveries["delivery_date_missing"] = deliveries["actual_delivery_date"].isna()
print(deliveries["delivery_date_missing"].sum())    # expect 20

20


In [7]:
for name, df in [("vendors", vendors), ("products", products),
                 ("orders", orders), ("deliveries", deliveries)]:
    missing_report(df, name)

--- vendors ---
No missing values
--- products ---
No missing values
--- orders ---
No missing values
--- deliveries ---
                      missing  missing_%
actual_delivery_date       20       0.34
